In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')


In [ ]:
print(stopwords.words('english'))

In [ ]:
simple_dictionary = pd.read_csv("corpus_with_frequency.csv")
simple_dictionary.head()

In [ ]:
# simple_dictionary = dict(simple_dictionary)
simple_dictionary_top2500 = simple_dictionary.iloc[:2500]
simple_dictionary_top2500 = simple_dictionary_top2500.rename(columns={'0':'word', '1': 'frequency'})
simple_dictionary_top2500.head()

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
articles = pd.read_csv('arts_books.csv', header=0, sep='^', engine='python', quoting=3)
articles

In [ ]:
# Count syllables
def count_syllables(word):
    word = word.lower()
    return max(1, len(re.findall(r'[aeiouy]+', word)))

# Count sentences
def count_sentences(text):
    return max(1, len(re.findall(r'[.!?]+', text)))

# Calculate SMOG index
def calculate_smog(text):
    words = text.split()
    polysyllable_count = sum(count_syllables(word) >= 3 for word in words)
    sentence_count = count_sentences(text)
    smog_index = 1.0430 * (30 * (polysyllable_count / sentence_count))**0.5 + 3.1291
    return smog_index

# Calculate FKGL index
def calculate_fkgl(text):
    sentences = text.split('.')
    num_sentences = len([sentence for sentence in sentences if sentence.strip() != ''])
    words = text.split()
    num_words = len(words)
    num_syllables = sum(count_syllables(word) for word in words)
    if num_sentences == 0 or num_words == 0:
        return 0
    return 0.39 * (num_words / num_sentences) + 11.8 * (num_syllables / num_words) - 15.59

# Calculate Our Score
def calculate_our_score(output_text, input_text, stop_words, simple_dictionary, model):
    input_word_tokens = word_tokenize(input_text)
    print('total number of words input', len(input_word_tokens), model)
    input_filtered_words = [w for w in input_word_tokens if not w.lower() in stop_words]
    print('number of meaningful words input', len(input_filtered_words), model)

    input_simple_words = [w for w in input_filtered_words if w in simple_dictionary['word'].values]
    print('number of simple words input', len(input_simple_words), model)
    input_meaningful_words = input_filtered_words
    S_input = len(input_simple_words) / (len(input_meaningful_words) if len(input_meaningful_words) > 0 else 1)
    print('S_input', S_input, model)

    output_word_tokens = word_tokenize(output_text)
    print('total number of words output', len(output_word_tokens), model)
    output_filtered_words = [w for w in output_word_tokens if not w.lower() in stop_words]
    print('number of meaningful words output', len(output_filtered_words), model)

    output_simple_words = [w for w in output_filtered_words if w in simple_dictionary['word'].values]
    print('number of simple words output', len(output_simple_words), model)
    output_meaningful_words = output_filtered_words
    S_output = len(output_simple_words) / (len(output_meaningful_words) if len(output_meaningful_words) > 0 else 1)
    print('S_output', S_output, model)
    return S_output - S_input

In [ ]:
file_path = ["arts_books.csv"]

In [ ]:
def evaluation(file_path):
    data = pd.read_csv(file_path, header=0)
    data['teacher'] = data['teacher'].astype(str) # Ensures that the NaN value in row 43 is treated as a string

    result = pd.DataFrame()
    # Apply all calculations to the relevant columns
    result['smog_original'] = data['original'].apply(calculate_smog)
    result['smog_student'] = data['student'].apply(calculate_smog)
    result['smog_teacher'] = data['teacher'].apply(calculate_smog)

    result['fkgl_original'] = data['original'].apply(calculate_fkgl)
    result['fkgl_student'] = data['student'].apply(calculate_fkgl)
    result['fkgl_teacher'] = data['teacher'].apply(calculate_fkgl)

    result['our_score_student'] = data.apply(lambda row: calculate_our_score(output_text=row['student'], input_text=row['original'], stop_words=set(stopwords.words('english')),simple_dictionary=simple_dictionary_top2500, model='student'), axis=1)
    result['our_score_teacher'] = data.apply(lambda row: calculate_our_score(output_text=row['teacher'], input_text=row['original'], stop_words=set(stopwords.words('english')),simple_dictionary=simple_dictionary_top2500, model='teacher'), axis=1)

        # Identify lowest and highest score names
    result['lowest_smog'] = result[['smog_original', 'smog_student', 'smog_teacher']].idxmin(axis=1)
    result['lowest_fkgl'] = result[['fkgl_original', 'fkgl_student', 'fkgl_teacher']].idxmin(axis=1)
    result['highest_our_score'] = result[['our_score_student', 'our_score_teacher']].idxmax(axis=1)

        # SMOG index descriptive statistics and counts
    lowest_smog_counts = result['lowest_smog'].value_counts()
    smog_descriptive_stats = result[['smog_original', 'smog_student', 'smog_teacher']].describe()

    # FKGL index descriptive statistics and counts
    lowest_fkgl_counts = result['lowest_fkgl'].value_counts()
    fkgl_descriptive_stats = result[['fkgl_original', 'fkgl_student', 'fkgl_teacher']].describe()

    # Our Score descriptive statistics and counts
    highest_our_score_counts = result['highest_our_score'].value_counts()
    our_score_descriptive_stats = result[['our_score_student', 'our_score_teacher']].describe()

    file_name = file_path.split('/')[-1]
    result.to_csv("./evaluation/evaluation_result_"+file_name)
    # Print results for easy viewing
    print("Lowest SMOG Counts:")
    print(lowest_smog_counts)
    print("\nSMOG Descriptive Statistics:")
    print(smog_descriptive_stats)

    print("\nLowest FKGL Counts:")
    print(lowest_fkgl_counts)
    print("\nFKGL Descriptive Statistics:")
    print(fkgl_descriptive_stats)

    print("\nHighest Our Score Counts:")
    print(highest_our_score_counts)
    print("\nOur Score Descriptive Statistics:")
    print(our_score_descriptive_stats)

    

In [ ]:
for path in file_path:
    evaluation(file_path=path)